# SEAS 6505 – Homework 6
**AI‑Driven Urban Traffic Optimization Prototype**  
**Student:** Christopher Paul

This notebook turns my slide deck into a working proof‑of‑concept. I simulate traffic on a small network of intersections, train a graph‑based model to forecast congestion fifteen minutes ahead, and show that it beats a simple baseline. The goal is to give a potential city partner or VC a clear signal that this idea is technically feasible.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(42)
np.random.seed(42)

print('PyTorch version:', torch.__version__)

## 1. Traffic network as a graph
I start with a small road network that behaves like a city corridor. Each node is an intersection, and each edge is a road segment between intersections. This keeps the prototype light but still lets me show why a graph model makes sense.

In [ ]:
# number of intersections
num_nodes = 12

# edges of a simple 3x4 grid (undirected)
edges = [
    (0,1),(1,2),(2,3),
    (4,5),(5,6),(6,7),
    (8,9),(9,10),(10,11),
    (0,4),(4,8),
    (1,5),(5,9),
    (2,6),(6,10),
    (3,7),(7,11)
]

A = torch.zeros(num_nodes, num_nodes)
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

# add self loops and normalize (GCN style)
A_hat = A + torch.eye(num_nodes)
deg = A_hat.sum(1)
D_inv_sqrt = torch.diag(1.0 / torch.sqrt(deg))
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

print('Adjacency shape:', A.shape)

## 2. Simulated traffic data
In a real deployment, this model would plug into DOT feeds, loop detectors, or connected‑vehicle data. For the prototype, I simulate traffic volumes with a daily pattern, rush‑hour spikes, neighbor influence, and random incidents that cause short term congestion. This is enough to show that the model can learn useful structure from the data.

In [ ]:
T = 500  # time steps (for example 500 x 15 minutes ≈ 5 days)
minutes_per_step = 15
steps_per_day = int(24 * 60 / minutes_per_step)

time = np.arange(T)
tod = time % steps_per_day
tod_sin = np.sin(2 * np.pi * tod / steps_per_day).astype(np.float32)
tod_cos = np.cos(2 * np.pi * tod / steps_per_day).astype(np.float32)

volumes = np.zeros((T, num_nodes), dtype=np.float32)
incidents = np.zeros((T, num_nodes), dtype=np.float32)

# initial traffic at t = 0
volumes[0] = 20 + 5 * np.random.randn(num_nodes)

for t in range(1, T):
    rush_am = (tod[t] >= 28) & (tod[t] <= 40)   # morning rush
    rush_pm = (tod[t] >= 70) & (tod[t] <= 90)   # evening rush
    rush_factor = 1.0 + 0.8 * rush_am + 1.0 * rush_pm

    prev = volumes[t - 1]
    neighbor_mean = (A.numpy() @ prev) / np.clip(A.numpy().sum(1), 1, None)

    base = 15 + 0.6 * prev + 0.3 * neighbor_mean \
           + 10 * np.sin(2 * np.pi * t / steps_per_day)

    noise = np.random.randn(num_nodes) * 3.0
    v = base * rush_factor + noise

    # random incidents that spike congestion
    if np.random.rand() < 0.05:
        k = np.random.randint(0, num_nodes)
        v[k] += 40
        incidents[t, k] = 1.0

    volumes[t] = np.clip(v, 0, None)

print('Simulated traffic series shape:', volumes.shape)

## 3. Building a supervised dataset
The task is to forecast congestion one step ahead at every intersection. I use the last four time steps for each node, plus time of day and an incident flag, to predict the next traffic volume. This gives a three dimensional tensor shaped as batch by node by feature.

In [ ]:
lookback = 4
F = lookback + 2 + 1  # four lagged volumes, sin, cos, incident flag
num_samples = T - lookback - 1

X = np.zeros((num_samples, num_nodes, F), dtype=np.float32)
y = np.zeros((num_samples, num_nodes), dtype=np.float32)

for idx in range(num_samples):
    t = lookback + idx
    for n in range(num_nodes):
        hist = volumes[t - lookback:t, n]
        feat = np.concatenate([
            hist,
            [tod_sin[t], tod_cos[t]],
            [incidents[t - 1, n]]
        ])
        X[idx, n, :] = feat
    y[idx] = volumes[t]

# simple scaling so the model has an easier job
y_scale = y.mean()
X[:, :, :lookback] /= y_scale
y /= y_scale

train_size = int(num_samples * 0.8)
X_train = torch.tensor(X[:train_size])
y_train = torch.tensor(y[:train_size])
X_test = torch.tensor(X[train_size:])
y_test = torch.tensor(y[train_size:])

print('Train samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])

## 4. Graph‑based forecasting model
To match the story from the slides, I use a simple graph neural network. Each layer mixes information from an intersection and its neighbors using the normalized adjacency matrix. The model predicts the next step of traffic volume at every node in one shot.

In [ ]:
class SimpleGNNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_lin = nn.Linear(in_dim, out_dim)
        self.neigh_lin = nn.Linear(in_dim, out_dim)

    def forward(self, X, A_norm):
        # X: [batch, nodes, features]
        neigh = torch.matmul(A_norm, X)
        return torch.relu(self.self_lin(X) + self.neigh_lin(neigh))


class TrafficGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=32):
        super().__init__()
        self.gnn1 = SimpleGNNLayer(in_dim, hidden_dim)
        self.gnn2 = SimpleGNNLayer(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, X, A_norm):
        h = self.gnn1(X, A_norm)
        h = self.gnn2(h, A_norm)
        out = self.out(h).squeeze(-1)
        return out


model = TrafficGNN(F, hidden_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Training loop
I train for a small number of epochs just to get a sensible fit. The focus here is not hyperparameter tuning. The real point is to show that the graph model learns patterns that a simple rule cannot see.

In [ ]:
num_epochs = 25

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train, A_norm)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_test = model(X_test, A_norm)
        val_loss = loss_fn(pred_test, y_test).item()

    if epoch == 1 or epoch % 5 == 0:
        print(f'Epoch {epoch:02d} | train MSE {loss.item():.4f} | val MSE {val_loss:.4f}')

# keep predictions for evaluation and plots
model.eval()
with torch.no_grad():
    preds = model(X_test, A_norm)

## 6. Evaluation versus a naïve baseline
As a sanity check, I compare the model to a very simple baseline that just repeats the last observed volume. If the model cannot beat that, the idea is not compelling. I report mean absolute error for both approaches and show the relative improvement.

In [ ]:
# baseline uses the most recent lagged volume
last_lag = X_test[:, :, 3]
baseline = last_lag

mae_model = (preds - y_test).abs().mean().item()
mae_baseline = (baseline - y_test).abs().mean().item()

print('Model MAE:   ', round(mae_model, 4))
print('Baseline MAE:', round(mae_baseline, 4))
improvement = (mae_baseline - mae_model) / mae_baseline * 100
print('Relative improvement vs baseline: {:.1f}%'.format(improvement))

# simple decision layer for the last test step
with torch.no_grad():
    last_pred = preds[-1] * y_scale
    last_true = y_test[-1] * y_scale

congestion_scores = last_pred.numpy()
top_k = 3
top_nodes = congestion_scores.argsort()[-top_k:][::-1]

print('\nTop intersections to optimize at the next interval:')
for rank, node_id in enumerate(top_nodes, start=1):
    print(
        f'{rank}. Node {node_id} | predicted volume ≈ {congestion_scores[node_id]:.1f} ' \
        f'(actual {last_true[node_id].item():.1f})'
    )

## 7. Visualizing the behavior
For a quick visual, I look at predicted versus actual traffic for one intersection and a heatmap for a subset of nodes. That gives sponsors an intuitive feel for how the model tracks congestion over time.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    true_test = y_test.numpy() * y_scale
    pred_test = preds.numpy() * y_scale

# choose one node
node = 2
plt.figure(figsize=(8, 4))
plt.plot(true_test[:, node], label='actual')
plt.plot(pred_test[:, node], label='predicted')
plt.xlabel('Test time step')
plt.ylabel('Traffic volume')
plt.title('Predicted vs actual traffic volume at node 2')
plt.legend()
plt.tight_layout()
plt.show()

# heatmap for a subset of nodes over the last part of the test window
timesteps = 48
nodes_to_plot = [0, 1, 2, 3, 4, 5]

plt.figure(figsize=(8, 4))
subset = true_test[-timesteps:, nodes_to_plot].T
plt.imshow(subset, aspect='auto')
plt.colorbar(label='Traffic volume')
plt.yticks(range(len(nodes_to_plot)), nodes_to_plot)
plt.xlabel('Last test time steps')
plt.ylabel('Node index')
plt.title('Actual traffic volume heatmap for selected nodes')
plt.tight_layout()
plt.show()

## 8. Technical analysis
This prototype shows that a simple graph neural network can forecast short term congestion on a small synthetic network. The core ideas are:

1. Model the city as a graph, where intersections are nodes and roads are edges.  
2. Encode realistic patterns like rush hours, neighbor influence, and random incidents in the simulated data.  
3. Use node features that combine recent volumes, time of day, and an incident indicator.  
4. Apply a two layer graph model to propagate information across the network and predict the next step for all nodes in parallel.  
5. Compare the model against a simple "repeat the last value" baseline and track mean absolute error.  

In my runs, the graph model consistently improves on the baseline error, which means it is learning structure beyond a trivial rule. The code is intentionally lightweight so it can be extended later to deeper architectures, longer horizons, or real data feeds.

## 9. Business analysis and next steps
From a sponsor or VC perspective, this notebook is a working proof that the traffic optimization platform is not just slides. Even on simulated data, the model learns to spot where the network is headed and surfaces the top intersections that need attention in the next fifteen minutes. That same logic can drive signal timing adjustments, dynamic message signs, or recommendations for where to deploy human operators.

The commercial upside is tied to measurable improvements in travel time, delay, and emissions. If this prototype delivers even a small improvement at corridor scale, a city can monetize it through reduced congestion costs and better reliability for freight and commuters. On the roadmap, the next steps are:

1. Replace the synthetic generator with real detector or probe data from a partner city.  
2. Extend the model to multi step forecasting so it can support rolling horizon control.  
3. Integrate the predictions into a dashboard that operations staff can use without touching code.  

This gives investors a clear path from proof‑of‑concept to a deployable product with recurring value for transportation agencies.